# 05 · Flight delay model
Binary classification: will a flight arrive more than 15 minutes late?

Four active models are compared on AUC, F1 and accuracy:

1. Logistic regression (baseline)
2. Logistic regression with class weights
3. Gradient-boosted trees
4. Logistic regression tuned with 3-fold cross-validation

Three more models are included **commented out** in the optional section.

Rules this notebook follows (each one fixed a real failure):

* `StringIndexer(handleInvalid="keep")`. `"skip"` silently drops rows.
* Every derived column (`DepHour`, `label`, `classWeight`) is created **before** `randomSplit`.
* `GBTClassifier` is not given `probabilityCol`; it does not accept it.

> Uses the classic BTS column names (see notebook 04). Edit the config cell if yours differ.

In [ ]:
from pyspark.sql import SparkSession

# Catalog config (REST catalog + MinIO) comes from spark-defaults.conf.
spark = SparkSession.builder.appName("05-flight-delay").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| UI:", spark.sparkContext.uiWebUrl)

from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.ml.classification import GBTClassifier, LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# ── Config ───────────────────────────────────────────────────────────────────
TABLE = "local.airline.flights"
SAMPLE_FRACTION = 0.1      # use 1.0 for everything; 0.1 keeps a laptop responsive
DELAY_THRESHOLD = 15       # minutes late at arrival counts as "delayed"
SEED = 42
CATEGORICAL = ["UniqueCarrier", "Origin", "Dest"]
NUMERIC = ["Month", "DayOfWeek", "DepHour", "Distance"]

## Build the modelling dataset
All derived columns are created here, before the split.

In [ ]:
base = (
    spark.table(TABLE)
    .filter((F.col("Cancelled") == 0) & (F.col("Diverted") == 0))
    .withColumn("DepHour", F.floor(F.col("CRSDepTime") / 100).cast("int"))
    .withColumn("label", (F.col("ArrDelay") > DELAY_THRESHOLD).cast("double"))
    .dropna(subset=["ArrDelay", "CRSDepTime", "Distance", "Month", "DayOfWeek"] + CATEGORICAL)
)
if SAMPLE_FRACTION < 1.0:
    base = base.sample(fraction=SAMPLE_FRACTION, seed=SEED)

pos_rate = base.agg(F.avg("label")).first()[0]
print(f"delayed share: {pos_rate:.1%}")

# Balanced class weights: each class contributes equally to the loss.
base = base.withColumn(
    "classWeight",
    F.when(F.col("label") == 1.0, F.lit(1.0 / (2.0 * pos_rate))).otherwise(F.lit(1.0 / (2.0 * (1.0 - pos_rate)))),
)
data = base.select(CATEGORICAL + NUMERIC + ["label", "classWeight"]).cache()

train, test = data.randomSplit([0.8, 0.2], seed=SEED)
print(f"train {train.count():,} | test {test.count():,}")

## Shared feature pipeline stages

In [ ]:
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in CATEGORICAL]
encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in CATEGORICAL],
    outputCols=[f"{c}_vec" for c in CATEGORICAL],
    handleInvalid="keep",
)
assembler = VectorAssembler(
    inputCols=NUMERIC + [f"{c}_vec" for c in CATEGORICAL],
    outputCol="features",
    handleInvalid="keep",
)
feature_stages = indexers + [encoder, assembler]

auc_eval = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
f1_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
acc_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

results = []
def evaluate(name, model):
    pred = model.transform(test)
    results.append({"model": name, "AUC": auc_eval.evaluate(pred),
                    "F1": f1_eval.evaluate(pred), "Accuracy": acc_eval.evaluate(pred)})
    print(f"✅ {name:<32} AUC={results[-1]['AUC']:.4f}  F1={results[-1]['F1']:.4f}  Acc={results[-1]['Accuracy']:.4f}")
    return pred

## Model 1 · Logistic regression (baseline)

In [ ]:
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)
m1 = Pipeline(stages=feature_stages + [lr]).fit(train)
_ = evaluate("LR baseline", m1)

## Model 2 · Logistic regression with class weights

In [ ]:
lr_w = LogisticRegression(featuresCol="features", labelCol="label", weightCol="classWeight", maxIter=20)
m2 = Pipeline(stages=feature_stages + [lr_w]).fit(train)
_ = evaluate("LR class-weighted", m2)

## Model 3 · Gradient-boosted trees
No `probabilityCol` argument: `GBTClassifier` does not accept it.

In [ ]:
gbt = GBTClassifier(featuresCol="features", labelCol="label", maxIter=20, maxDepth=5, seed=SEED)
m3 = Pipeline(stages=feature_stages + [gbt]).fit(train)
_ = evaluate("GBT", m3)

## Model 4 · Logistic regression, cross-validated

In [ ]:
lr_cv = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)
pipe_cv = Pipeline(stages=feature_stages + [lr_cv])
grid = ParamGridBuilder().addGrid(lr_cv.regParam, [0.0, 0.01, 0.1]).build()
cv = CrossValidator(estimator=pipe_cv, estimatorParamMaps=grid, evaluator=auc_eval,
                    numFolds=3, parallelism=2, seed=SEED)
m4 = cv.fit(train)
_ = evaluate("LR cross-validated", m4)
print("best regParam:", m4.bestModel.stages[-1].getRegParam())

## Optional models (commented out)
Uncomment to run. They are kept in the notebook on purpose, not deleted.

In [ ]:
# from pyspark.ml.classification import RandomForestClassifier, DecisionTreeClassifier, LinearSVC
#
# rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=8, seed=SEED)
# m5 = Pipeline(stages=feature_stages + [rf]).fit(train)
# _ = evaluate("Random forest", m5)
#
# dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=8, seed=SEED)
# m6 = Pipeline(stages=feature_stages + [dt]).fit(train)
# _ = evaluate("Decision tree", m6)
#
# svc = LinearSVC(featuresCol="features", labelCol="label", maxIter=20)
# m7 = Pipeline(stages=feature_stages + [svc]).fit(train)
# _ = evaluate("Linear SVM", m7)

## Compare

In [ ]:
%matplotlib inline
import pandas as pd

summary = pd.DataFrame(results).set_index("model").sort_values("AUC", ascending=False)
display(summary.round(4))
summary[["AUC", "F1", "Accuracy"]].plot(kind="barh", figsize=(8, 3.5))